In [2]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import sklearn

torch.set_num_threads(1)
torch.set_num_interop_threads(1)

In [4]:
input_df = pd.read_csv('cleaned_data/input_data_2024-25.csv')
target_df = pd.read_csv('cleaned_data/target_data_2024-25.csv')

In [5]:
combined_df = pd.merge(input_df, target_df, how='inner', on=['personId', 'gameId', 'home'])
combined_df = combined_df.dropna()

In [6]:
n_starters = 5

stat_cols = [
    'h_points', 
    'h_assists', 
    'h_reboundsTotal',
    'h_blocks', 
    'h_steals',
    'h_fieldGoalsAttempted', 
    'h_fieldGoalsMade', 
    'h_threePointersAttempted',
    'h_threePointersMade', 
    'h_freeThrowsAttempted', 
    'h_freeThrowsMade',
    'h_reboundsDefensive', 
    'h_reboundsOffensive', 
    'h_foulsPersonal', 
    'h_turnovers', 
    'h_numMinutes'
]

target_cols = [
    'points',
    'assists',
    'reboundsTotal'
]

In [7]:
max_players = combined_df.groupby('gameId').size().max()

In [8]:
combined_df['gameDateTimeEst'] = pd.to_datetime(combined_df['gameDateTimeEst'])
combined_df = combined_df.sort_values('gameDateTimeEst')

cutoff_idx = int(len(combined_df['gameDateTimeEst'].unique()) * 0.8)
cutoff_date = combined_df['gameDateTimeEst'].unique()[cutoff_idx]

train_df = combined_df[combined_df['gameDateTimeEst'] <= cutoff_date]
val_df = combined_df[combined_df['gameDateTimeEst'] > cutoff_date]

In [9]:
def build_input(
    df, 
    n_starters=n_starters, 
    minutes_col='h_numMinutes',
    stat_cols=None,
    target_cols=None
):
    flag = True

    home_df = df[df['home'] == 1].sort_values(minutes_col, ascending=False)
    if len(home_df) < n_starters+1:
        flag = False
    else:
        keep = home_df.head(n_starters)
        rest = home_df.iloc[n_starters:]
        weights = rest[minutes_col].values

        weighted_stats = {
            col: np.average(rest[col], weights=weights) for col in stat_cols
        }

        bench_row = {
            **{c: rest.iloc[0][c] for c in ['gameId', 'home']},
            **weighted_stats
        }

        pooled_df = pd.concat(
            [keep, pd.DataFrame([bench_row])],
            ignore_index=True
        )

        home_feats = pooled_df[stat_cols].values

    away_df = df[df['home'] == 0].sort_values(minutes_col, ascending=False)
    if len(away_df) < n_starters+1:
        flag = False
    else:
        keep = away_df.head(n_starters)
        rest = away_df.iloc[n_starters:]
        weights = rest[minutes_col].values

        weighted_stats = {
            col: np.average(rest[col], weights=weights) for col in stat_cols
        }

        bench_row = {
            **{c: rest.iloc[0][c] for c in ['gameId', 'home']},
            **weighted_stats
        }

        pooled_df = pd.concat(
            [keep, pd.DataFrame([bench_row])],
            ignore_index=True
        )

        away_feats = pooled_df[stat_cols].values
    
    if flag:
        team_vec = np.concatenate([home_feats, away_feats], axis=0).ravel()
    else:
        team_vec = None
    
    player_feats = df[stat_cols + ['home']].values
    player_targets = df[target_cols].values
    line_cols = []
    for col in target_cols:
        line_cols.append('h_' + col)
    player_lines = df[line_cols].values
    num_players = player_feats.shape[0]

    pad_len = max_players - num_players
    if pad_len > 0:
        padding = np.zeros((pad_len, len(stat_cols)+1))
        padding_target = np.zeros((pad_len, len(target_cols)))
        player_feats = np.vstack([player_feats, padding])
        player_targets = np.vstack([player_targets, padding_target])
        player_lines = np.vstack([player_lines, padding_target])
    
    mask = np.zeros(max_players)
    mask[:num_players] = 1

    return [team_vec], player_feats, player_targets, mask, flag, player_lines

In [10]:
def build_dataset(df):
    X_team = []
    X_players = []
    player_masks = []
    Y = []
    player_lines = []

    for _, game in df.groupby('gameId'):
        team_vec, player_feats, player_targets, mask, flag, lines = build_input(game, n_starters, 'h_numMinutes', stat_cols, target_cols)
        if flag:
            X_team.extend(team_vec)
            X_players.append(player_feats)
            Y.append(player_targets)
            player_masks.append(mask)
            player_lines.append(lines)

    X_team = torch.tensor(np.stack(X_team), dtype=torch.float32)
    X_players = torch.tensor(np.stack(X_players), dtype=torch.float32)
    Y = torch.tensor(np.stack(Y), dtype=torch.float32)
    player_masks = torch.tensor(np.stack(player_masks), dtype=torch.float32)
    player_lines = torch.tensor(np.stack(player_lines), dtype=torch.float32)

    return X_team, X_players, player_masks, Y, player_lines

X_team_train, X_players_train, player_masks_train, Y_train, lines_train = build_dataset(train_df)
X_team_val, X_players_val, player_masks_val, Y_val, lines_val = build_dataset(val_df)

Y_mean = Y_train.mean(dim=(0,1))
Y_std = Y_train.std(dim=(0,1)) + 1e-6
Y_train = (Y_train - Y_mean) / Y_std
Y_val = (Y_val - Y_mean) / Y_std

In [11]:
print(X_team_train.shape)
print(X_players_train.shape)
print(player_masks_train.shape)
print(Y_train.shape)
print(lines_train.shape)
print(X_team_val.shape)
print(X_players_val.shape)
print(player_masks_val.shape)
print(Y_val.shape)
print(lines_val.shape)

torch.Size([948, 192])
torch.Size([948, 28, 17])
torch.Size([948, 28])
torch.Size([948, 28, 3])
torch.Size([948, 28, 3])
torch.Size([259, 192])
torch.Size([259, 28, 17])
torch.Size([259, 28])
torch.Size([259, 28, 3])
torch.Size([259, 28, 3])


In [12]:
class GameEncoder(nn.Module):
    def __init__(self, input_dim, h_dim=64, latent_dim=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 2*h_dim),
            nn.ReLU(),
            nn.Linear(2*h_dim, h_dim),
            nn.ReLU()
        )
        self.mu = nn.Linear(h_dim, latent_dim)
        self.logvar = nn.Linear(h_dim, latent_dim)
    
    def forward(self, x):
        h = self.net(x)
        return self.mu(h), self.logvar(h)

class PlayerDecoder(nn.Module):
    def __init__(self, latent_dim, player_dim, h_dim=32, output_dim=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim + player_dim, h_dim),
            nn.ReLU(),
            nn.Linear(h_dim, output_dim)
        )
    
    def forward(self, z, player_feats):
        z = z.unsqueeze(1).expand(-1, player_feats.size(1), -1)
        x = torch.cat([z, player_feats], dim=-1)
        return self.net(x)

def reparameterize(mu, logvar):
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    return mu + eps * std

In [13]:
latent_dim = 16
lr = 1e-3

encoder = GameEncoder(input_dim=192, h_dim=64, latent_dim=latent_dim)
decoder = PlayerDecoder(latent_dim=latent_dim, player_dim=X_players_train.shape[2], h_dim=32, output_dim=Y_train.shape[2])

optimizer = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=lr)

In [14]:
class NBADataset(torch.utils.data.Dataset):
    def __init__(self,  X_team, X_players, Y, mask, lines):
        self.X_team = X_team
        self.X_players = X_players
        self.Y = Y
        self.mask = mask
        self.lines = lines
    
    def __len__(self):
        return len(self.X_team)
    
    def __getitem__(self, idx):
        return (
            self.X_team[idx],
            self.X_players[idx],
            self.Y[idx],
            self.mask[idx],
            self.lines[idx]
        )

In [15]:
train_dataset = NBADataset(X_team_train, X_players_train, Y_train, player_masks_train, lines_train)
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    persistent_workers=False,
    pin_memory=False
)

val_dataset = NBADataset(X_team_val, X_players_val, Y_val, player_masks_val, lines_val)
val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    persistent_workers=False,
    pin_memory=False
)

In [16]:
def masked_mse(pred, target, mask):
    diff = (pred - target) ** 2
    diff = diff * mask.unsqueeze(-1)
    return diff.sum() / (mask.sum() * pred.size(-1) + 1e-8)

def kl_divergence(mu, logvar):
    return -0.5 * torch.mean(
        torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)
    )

def train_step(X_team, X_players, Y, mask, beta=0.001):
    optimizer.zero_grad()

    mu, logvar = encoder(X_team)
    z = reparameterize(mu, logvar)
    
    preds = decoder(z, X_players)

    recon = masked_mse(preds, Y, mask)
    kl = kl_divergence(mu, logvar)

    loss = recon + beta * kl
    loss.backward()
    optimizer.step()

    return {
        "loss": loss.item(),
        "recon": recon.item(),
        "kl": kl.item()
    }


In [17]:
encoder.train()
decoder.train()

num_epochs = 15
beta = 0.001

for epoch in range(num_epochs):
    metrics = {"loss": 0, "recon": 0, "kl": 0}

    for X_t, X_p, Y_b, mask_b, lines_b in train_loader:
        out = train_step(X_t, X_p, Y_b, mask_b, beta=beta)

        for k in metrics:
            metrics[k] += out[k]

    for k in metrics:
        metrics[k] /= len(train_loader)

    print(
        f"Epoch {epoch:03d} | "
        f"Loss {metrics['loss']:.4f} | "
        f"Recon {metrics['recon']:.4f} | "
        f"KL {metrics['kl']:.4f}"
    )

Epoch 000 | Loss 1.0109 | Recon 0.9619 | KL 48.9882
Epoch 001 | Loss 0.6978 | Recon 0.6568 | KL 40.9805
Epoch 002 | Loss 0.6144 | Recon 0.5899 | KL 24.4812
Epoch 003 | Loss 0.5712 | Recon 0.5520 | KL 19.2197
Epoch 004 | Loss 0.5500 | Recon 0.5338 | KL 16.2153
Epoch 005 | Loss 0.5341 | Recon 0.5196 | KL 14.5005
Epoch 006 | Loss 0.5242 | Recon 0.5107 | KL 13.5364
Epoch 007 | Loss 0.5229 | Recon 0.5101 | KL 12.7722
Epoch 008 | Loss 0.5168 | Recon 0.5042 | KL 12.5804
Epoch 009 | Loss 0.5162 | Recon 0.5044 | KL 11.7979
Epoch 010 | Loss 0.5121 | Recon 0.5006 | KL 11.5132
Epoch 011 | Loss 0.5088 | Recon 0.4981 | KL 10.7186
Epoch 012 | Loss 0.5113 | Recon 0.5003 | KL 11.0077
Epoch 013 | Loss 0.5088 | Recon 0.4983 | KL 10.4680
Epoch 014 | Loss 0.5071 | Recon 0.4965 | KL 10.6146


In [18]:
encoder.eval()
decoder.eval()

num_samples = 1
total_loss = 0.0
total_recon = 0.0
total_kl = 0.0
num_batches = 0

with torch.no_grad():
    for X_t, X_p, Y_b, mask_b, lines_b in val_loader:
        mu, logvar = encoder(X_t)

        kl = kl_divergence(mu, logvar)

        recon_samples = []
        for n in range(num_samples):
            z = reparameterize(mu, logvar)
            preds = decoder(z, X_p)
            recon_samples.append(masked_mse(preds, Y_b, mask_b))
        
        recon = torch.stack(recon_samples).mean()

        loss = recon + beta * kl

        total_loss += loss.item()
        total_recon += recon.item()
        total_kl += kl.item()
        num_batches += 1

avg_loss = total_loss / num_batches
avg_recon = total_recon / num_batches
avg_kl = total_kl / num_batches

print(f"Stochastic Validation Loss: {avg_loss:.4f} (Recon: {avg_recon:.4f}, KL: {avg_kl:.4f})")

Stochastic Validation Loss: 0.5480 (Recon: 0.5396, KL: 8.4195)


In [19]:
encoder.eval()
decoder.eval()

num_samples = 10
all_samples = []

with torch.no_grad():
    for X_t, X_p, Y_b, mask_b, lines in val_loader:
        mu, logvar = encoder(X_t)

        batch_preds = []
        for n in range(num_samples):
            z = reparameterize(mu, logvar)

            preds = decoder(z, X_p)
            preds = (preds * Y_std) + Y_mean

            over = (preds > lines).int()
            batch_preds.append(over)

        over = torch.stack(batch_preds, dim=0)
        _, batch_size, n_players, n_stats = over.shape
        A = over.permute(1, 2, 3, 0)
        A = A.reshape(batch_size, n_players * n_stats, num_samples)
        S = A.sum(dim=2)
        OO = A @ A.transpose(1, 2)
        OU = S.unsequeeze(2) - OO
        UO = S.unsqueeze(1) - OO
        UU = num_samples - OO - OU - UO
        print(OO.shape)
        print(OU.shape)
        print(UO.shape)
        print(UU.shape)



AttributeError: 'Tensor' object has no attribute 'unsequeeze'